# Usage | 3. Existing bike network
This notebook explains how GrowBikeNet can extend an existing bike network.

**Parameters covered**: `existing_network_spacing`

We start every Usage notebook with the standard way of importing GrowBikeNet:

In [1]:
import growbikenet as gbn

## Adding seed points on the existing bike network

So far GrowBikeNet was executed with the default parameter setting `existing_network_spacing=None`, which instructed GrowBikeNet to ignore existing bicycle infrastructure. This works for most cities, as existing infrastructure is usually negligible and one might as well just start from scratch. However, there are some cities with an already existing substantial network which would be useful to incorporate into the growth process. By calling GrowBikeNet with the parameter `existing_network_spacing='auto'` or with a positive integer, it will do exactly that.

In this case, the process of generating seed points is amended beforehand:
- Consider all network components of the existing bike network that have a minimum length. This ensures that tiny, insignificant pieces are ignored.
- On these components, choose a random first seed point.
- Choose the closest seed point on the components that is at least `existing_network_spacing` meters away. The `'auto'` option automatically chooses a recommended distance, at 50% of the `seed_point_grid_spacing`.
- Proceed with the previous step until no more seed points can be placed on the components.
- Now generate all the other seed points as usual, but do not consider seed points that are too close to already existing seed points.

Let us run GrowBikeNet on Athens, Greece with the `existing_network_spacing='auto'` option and observe the results:

In [2]:
edges_ordered = gbn.growbikenet("Municipality of Athens",
    existing_network_spacing='auto',)

RUNNING GROWBIKENET FOR CITY: Municipality of Athens
betweenness | auto | from existing bike network 
----------------------------------------------╮


Exporting data         : 100%|████████████████| 1/1 [00:00<00:00, 14.73step/s]

----------------------------------------------╯
Data exported to ./results/
----------------------------------------------
FINISHED IN 0:01:25


The existing bike network is saved as multilinestring into the first row of the resulting geodataframe with several entries being `None`:

In [3]:
edges_ordered.head()

,betweenness,geometry,source,target,ordering,length,length_cumulative
0,None,"MULTILINESTRING ((23.7319 37.97968, 23.73195 3...",None,None,0,27201,27201
1,0.128205,"LINESTRING (23.72236 37.9984, 23.72232 37.9979...",358483920.0,250663738.0,1,1818,29019
2,0.097436,"MULTILINESTRING ((23.72348 37.97705, 23.72368 ...",6707879950.0,11270494938.0,2,441,29461
3,0.095833,"LINESTRING (23.74462 37.97177, 23.74424 37.971...",7229807073.0,251110530.0,3,974,30436
4,0.092308,"LINESTRING (23.75267 37.99428, 23.75293 37.994...",370604212.0,360398103.0,4,2094,32530


To visualize the outcome, we plot first the existing bike network (first row) in blue, then the grown network (all other rows) in green. To add layer control in the top right of the map, we import folium:

In [4]:
import folium
viz = edges_ordered.iloc[:1].explore(
    tiles="CartoDB Positron",
    style_kwds={"weight": 2, "color": "#9999cc"},
    name="Existing bike network",
)
viz = edges_ordered.iloc[1:].explore(
    m=viz, 
    style_kwds={"weight": 3, "color": "#096a51"},
    name="Grown bike network",
)
folium.LayerControl().add_to(viz)
viz

Note how the short existing pieces in the northeast are ignored, but the other big enough components are incorporated into the growth process.

## Comparing with growth from scratch

Let us add the outcome from growth from scratch (without the existing network) in orange to see the difference:

In [5]:
edges_ordered_from_scratch = gbn.growbikenet("Municipality of Athens")

RUNNING GROWBIKENET FOR CITY: Municipality of Athens
betweenness | auto | from scratch
----------------------------------------------╮


Exporting data         : 100%|████████████████| 1/1 [00:00<00:00, 37.84step/s]

----------------------------------------------╯
Data exported to ./results/
----------------------------------------------
FINISHED IN 0:00:05


In [6]:
viz = edges_ordered.iloc[:1].explore(
    tiles="CartoDB Positron",
    style_kwds={"weight": 2, "color": "#9999cc"},
    name="Existing bike network",
)
viz = edges_ordered_from_scratch.explore(
    m=viz, 
    style_kwds={"weight": 3, "color": "#f19730"},
    name="Grown bike network (from scratch)",
)
viz = edges_ordered.iloc[1:].explore(
    m=viz, 
    style_kwds={"weight": 3, "color": "#096a51"},
    name="Grown bike network (with existing network)",
)
folium.LayerControl().add_to(viz)
viz

In general, the network which accounts for existing infrastructure will be longer than the one grown from scratch, in this case

In [7]:
int((edges_ordered.iloc[-1].length_cumulative-
    edges_ordered.iloc[0].length_cumulative)/1000)

145

kilometers compared to

In [8]:
int((edges_ordered_from_scratch.iloc[-1].length_cumulative)/1000)

125

kilometers, as seed points are generated more densely.